# Pawn counting

Project to explore/visualise chess pawn structures

In [2]:
import polars as pl
import numpy as np
import altair as alt

In [3]:
type Position = tuple[np.uint64, np.uint64]
"""
Pawn structure position --- a configuration of White and Black pawns on a board of size
BOARD_WIDTH * BOARD_DEPTH

Consists of two u64 bitmasks, the first for White pawns and the second for Black.

The least bit of each bitmask is the bottom-left square for White (a1), counting up in
file-major order (a1, a2, ..., a8, b1, ...) to the top-right square for White (h8).

If the board is smaller than 8x8, the board crops around bottom-left square for White
(so the bottom-left square remains a1), and the unused squares are empty.
"""

BOARD_WIDTH = 4
BOARD_DEPTH = 4

assert 0 <= BOARD_WIDTH <= 8
assert 0 <= BOARD_DEPTH <= 8

In [4]:
def position_ndarray(pos: Position) -> tuple[np.ndarray, np.ndarray]:
    def to_array(mask):
        arr = np.zeros((BOARD_WIDTH, BOARD_DEPTH), dtype=bool)
        for f in range(BOARD_WIDTH):
            for r in range(BOARD_DEPTH):
                arr[f, r] = (mask >> (f * 8 + r)) & 1
        return arr

    white, black = pos
    return to_array(white), to_array(black)

def init_position() -> Position:
    white = 0
    black = 0
    for f in range(BOARD_WIDTH):
        white |= 1 << (f * 8)
        black |= 1 << (f * 8 + BOARD_DEPTH - 1)
    return np.uint64(white), np.uint64(black)

def rand_position(max_pawns_per_side: int | None = None) -> Position:
    white = 0
    black = 0
    white_count = 0
    black_count = 0
    for f in range(BOARD_WIDTH):
        for r in range(BOARD_DEPTH):
            bit = 1 << (f * 8 + r)
            choice = np.random.randint(3)
            if choice == 1 and (max_pawns_per_side is None or white_count < max_pawns_per_side):
                white |= bit
                white_count += 1
            elif choice == 2 and (max_pawns_per_side is None or black_count < max_pawns_per_side):
                black |= bit
                black_count += 1
    return np.uint64(white), np.uint64(black)

def pawns_as_frame(pos: Position) -> pl.DataFrame:
    colour = pl.Enum(["White", "Black"])
    rows = []
    for name, mask in zip(["White", "Black"], pos):
        for f in range(BOARD_WIDTH):
            for r in range(BOARD_DEPTH):
                if (mask >> (f * 8 + r)) & 1:
                    rows.append({"rank": r + 1, "file": f + 1, "colour": name})
    return pl.DataFrame(rows, schema={"rank": pl.Int64, "file": pl.Int64, "colour": colour})

In [13]:
def position_chart(pos: Position) -> alt.Chart:
    _PAWN_COLOURS = {
        "domain": ["White", "Black"],
        "range": ["white", "black"],
    }
    _FILE_DOMAIN = list(range(1, BOARD_WIDTH + 1))
    _RANK_DOMAIN = list(range(1, BOARD_DEPTH + 1))

    pawns = pawns_as_frame(pos)
    return (
        alt.Chart(pawns)
        .mark_circle(size=100)
        .encode(
            alt.X("file:O").scale(domain=_FILE_DOMAIN),
            alt.Y("rank:O").scale(domain=_RANK_DOMAIN),
            alt.Color("colour:N")
            #
            .scale(**_PAWN_COLOURS),  # type: ignore
        )
        .properties(width=150, height=150)
    )

position_chart(rand_position(max_pawns_per_side=BOARD_WIDTH))

alt.Chart(...)